In [ ]:
"""
Lab Task 2: Loan Default Prediction using Naive Bayes Classifier
Predicting customers who have NOT fully paid their loan.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc)

# ─────────────────────────────────────────
# 1. Create Realistic Loan Dataset
# ─────────────────────────────────────────
np.random.seed(42)
n = 9578

credit_policy    = np.random.choice([0, 1], n, p=[0.20, 0.80])
purpose          = np.random.choice(['debt_consolidation','credit_card','home_improvement',
                                      'other','major_purchase','small_business','educational'], n)
int_rate         = np.round(np.random.uniform(0.06, 0.24, n), 4)
installment      = np.round(np.random.uniform(50, 900, n), 2)
log_annual_inc   = np.round(np.random.normal(11.0, 0.7, n), 4)
dti              = np.round(np.random.uniform(0, 35, n), 2)
fico             = np.random.randint(612, 827, n)
days_credit_line = np.random.randint(730, 6000, n)
revol_bal        = np.random.randint(0, 120000, n)
revol_util       = np.round(np.random.uniform(0, 119, n), 1)
inq_last_6mths   = np.random.randint(0, 9, n)
delinq_2yrs      = np.random.choice([0,1,2,3], n, p=[0.80,0.12,0.05,0.03])
pub_rec          = np.random.choice([0,1,2], n, p=[0.90,0.08,0.02])

# Target: not.fully.paid  (1 = defaulted, 0 = paid)
# Higher int_rate, lower fico, lower credit_policy → higher chance of default
default_prob = (0.15
                + 0.15 * (int_rate > 0.15)
                + 0.10 * (fico < 680)
                + 0.08 * (credit_policy == 0)
                + 0.05 * (dti > 25)
                - 0.05 * (log_annual_inc > 11.5))
default_prob = np.clip(default_prob, 0.05, 0.75)
not_fully_paid = np.array([np.random.binomial(1, p) for p in default_prob])

df = pd.DataFrame({
    'credit.policy'    : credit_policy,
    'purpose'          : purpose,
    'int.rate'         : int_rate,
    'installment'      : installment,
    'log.annual.inc'   : log_annual_inc,
    'dti'              : dti,
    'fico'             : fico,
    'days.with.cr.line': days_credit_line,
    'revol.bal'        : revol_bal,
    'revol.util'       : revol_util,
    'inq.last.6mths'   : inq_last_6mths,
    'delinq.2yrs'      : delinq_2yrs,
    'pub.rec'          : pub_rec,
    'not.fully.paid'   : not_fully_paid
})

# ─────────────────────────────────────────
# 2. Data Exploration
# ─────────────────────────────────────────
print("=" * 65)
print("        LAB TASK 2: LOAN DEFAULT PREDICTION")
print("=" * 65)

print(f"\n  Dataset Shape  : {df.shape}")
print(f"  Features       : {df.shape[1]-1}")
print(f"  Target         : not.fully.paid")

print("\n── Data Types ──")
print(df.dtypes)

print("\n── First 5 Rows ──")
print(df.head())

print("\n── Missing Values ──")
print(df.isnull().sum())

print("\n── Statistical Summary ──")
print(df.describe().round(3))

print("\n── Target Distribution ──")
vc = df['not.fully.paid'].value_counts()
print(f"  Fully Paid (0)    : {vc[0]}  ({vc[0]/len(df)*100:.1f}%)")
print(f"  Not Fully Paid (1): {vc[1]}  ({vc[1]/len(df)*100:.1f}%)")

print("\n── Loan Purpose Counts ──")
print(df['purpose'].value_counts())

print("\n── Default Rate by Purpose ──")
print(df.groupby('purpose')['not.fully.paid'].mean().sort_values(ascending=False).round(3))

print("\n── Correlation with Target ──")
corr = df.drop(columns=['purpose']).corr()['not.fully.paid'].sort_values(ascending=False)
print(corr.round(4))

# ─────────────────────────────────────────
# 3. Preprocessing
# ─────────────────────────────────────────
le = LabelEncoder()
df['purpose_enc'] = le.fit_transform(df['purpose'])
df_model = df.drop(columns=['purpose'])

X = df_model.drop(columns=['not.fully.paid'])
y = df_model['not.fully.paid']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
print(f"\n  Train samples : {len(X_train)}")
print(f"  Test  samples : {len(X_test)}")

# ─────────────────────────────────────────
# 4. Train Gaussian Naive Bayes
# ─────────────────────────────────────────
gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred     = gnb.predict(X_test)
y_prob     = gnb.predict_proba(X_test)[:, 1]
acc        = accuracy_score(y_test, y_pred)

print("\n" + "=" * 65)
print("  GAUSSIAN NAIVE BAYES — LOAN DEFAULT PREDICTION")
print("=" * 65)
print(f"  Accuracy : {acc:.4f}  ({acc*100:.2f}%)")
print("\n  Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Fully Paid','Not Fully Paid']))

# ─────────────────────────────────────────
# 5. Sample Predictions
# ─────────────────────────────────────────
print("\n── Sample Predictions (first 10 test samples) ──")
sample = X_test.head(10).copy()
sample['Actual']    = y_test.head(10).values
sample['Predicted'] = y_pred[:10]
sample['Prob_Default'] = y_prob[:10].round(3)
sample['Correct']   = sample['Actual'] == sample['Predicted']
print(sample[['fico','int.rate','dti','Actual','Predicted','Prob_Default','Correct']].to_string())

# ─────────────────────────────────────────
# 6. Visualizations
# ─────────────────────────────────────────
fig = plt.figure(figsize=(20, 16))
fig.patch.set_facecolor('#0e1117')

gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)
C0, C1 = '#4FC3F7', '#FF7043'

def darkax(ax, title=None, xlab=None, ylab=None):
    ax.set_facecolor('#1a1d27')
    ax.tick_params(colors='white', labelsize=8)
    ax.spines[:].set_color('#333')
    if title: ax.set_title(title, color='white', fontsize=10, fontweight='bold')
    if xlab:  ax.set_xlabel(xlab, color='#aaa', fontsize=8)
    if ylab:  ax.set_ylabel(ylab, color='#aaa', fontsize=8)

# (A) Target Distribution
ax0 = fig.add_subplot(gs[0, 0])
labels = ['Fully Paid', 'Not Fully Paid']
sizes  = [vc[0], vc[1]]
wedges, texts, autotexts = ax0.pie(sizes, labels=labels, autopct='%1.1f%%',
                                    colors=[C0, C1], startangle=140,
                                    textprops={'color':'white','fontsize':9})
for at in autotexts: at.set_color('white'); at.set_fontsize(10)
darkax(ax0, 'Target Distribution')

# (B) FICO by Default Status
ax1 = fig.add_subplot(gs[0, 1])
ax1.hist(df.loc[df['not.fully.paid']==0,'fico'], bins=30, alpha=0.7, color=C0, label='Fully Paid', edgecolor='white', lw=0.3)
ax1.hist(df.loc[df['not.fully.paid']==1,'fico'], bins=30, alpha=0.7, color=C1, label='Not Fully Paid', edgecolor='white', lw=0.3)
darkax(ax1, 'FICO Score Distribution', 'FICO Score', 'Count')
ax1.legend(fontsize=8, facecolor='#222', labelcolor='white')

# (C) Interest Rate by Default
ax2 = fig.add_subplot(gs[0, 2])
ax2.hist(df.loc[df['not.fully.paid']==0,'int.rate'], bins=30, alpha=0.7, color=C0, label='Fully Paid', edgecolor='white', lw=0.3)
ax2.hist(df.loc[df['not.fully.paid']==1,'int.rate'], bins=30, alpha=0.7, color=C1, label='Not Fully Paid', edgecolor='white', lw=0.3)
darkax(ax2, 'Interest Rate Distribution', 'Interest Rate', 'Count')
ax2.legend(fontsize=8, facecolor='#222', labelcolor='white')

# (D) Default Rate by Purpose
ax3 = fig.add_subplot(gs[1, 0:2])
dfr = df.groupby('purpose')['not.fully.paid'].mean().sort_values(ascending=True)
colors = [C1 if v > 0.20 else C0 for v in dfr.values]
ax3.barh(dfr.index, dfr.values*100, color=colors, edgecolor='white', lw=0.4)
darkax(ax3, 'Default Rate by Loan Purpose', 'Default Rate (%)', '')
for i, v in enumerate(dfr.values):
    ax3.text(v*100+0.3, i, f'{v*100:.1f}%', va='center', color='white', fontsize=8)

# (E) Confusion Matrix
ax4 = fig.add_subplot(gs[1, 2])
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Paid','Not Paid'])
disp.plot(ax=ax4, colorbar=False, cmap='coolwarm')
darkax(ax4, 'Confusion Matrix')
ax4.xaxis.label.set_color('white'); ax4.yaxis.label.set_color('white')
for text in ax4.texts: text.set_color('white')

# (F) ROC Curve
ax5 = fig.add_subplot(gs[2, 0])
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
ax5.plot(fpr, tpr, color=C1, lw=2, label=f'ROC AUC = {roc_auc:.3f}')
ax5.plot([0,1],[0,1], '--', color='gray', lw=1)
ax5.fill_between(fpr, tpr, alpha=0.2, color=C1)
darkax(ax5, 'ROC Curve', 'False Positive Rate', 'True Positive Rate')
ax5.legend(fontsize=9, facecolor='#222', labelcolor='white')

# (G) DTI vs Int Rate (colored by default)
ax6 = fig.add_subplot(gs[2, 1])
paid   = df[df['not.fully.paid']==0].sample(400)
deflt  = df[df['not.fully.paid']==1].sample(min(400, vc[1]))
ax6.scatter(paid['dti'],  paid['int.rate'],  c=C0, alpha=0.5, s=10, label='Fully Paid')
ax6.scatter(deflt['dti'], deflt['int.rate'], c=C1, alpha=0.5, s=10, label='Not Fully Paid')
darkax(ax6, 'DTI vs Interest Rate', 'Debt-to-Income Ratio', 'Interest Rate')
ax6.legend(fontsize=8, facecolor='#222', labelcolor='white')

# (H) Prediction probability distribution
ax7 = fig.add_subplot(gs[2, 2])
ax7.hist(y_prob[y_test==0], bins=30, alpha=0.7, color=C0, label='Actual Paid', edgecolor='white', lw=0.3)
ax7.hist(y_prob[y_test==1], bins=30, alpha=0.7, color=C1, label='Actual Default', edgecolor='white', lw=0.3)
darkax(ax7, 'Predicted Probability of Default', 'P(Default)', 'Count')
ax7.legend(fontsize=8, facecolor='#222', labelcolor='white')
ax7.axvline(0.5, color='yellow', linestyle='--', lw=1, label='threshold')

fig.suptitle('Lab Task 2 — Loan Default Prediction | Naive Bayes Classifier',
             color='white', fontsize=14, fontweight='bold', y=0.98)

plt.savefig('lab_task2_loan.png', dpi=150,
            bbox_inches='tight', facecolor=fig.get_facecolor())
print("\n[✔] Plot saved → lab_task2_loan.png")
plt.close()

        LAB TASK 2: LOAN DEFAULT PREDICTION

  Dataset Shape  : (9578, 14)
  Features       : 13
  Target         : not.fully.paid

── Data Types ──
credit.policy          int64
purpose               object
int.rate             float64
installment          float64
log.annual.inc       float64
dti                  float64
fico                   int64
days.with.cr.line      int64
revol.bal              int64
revol.util           float64
inq.last.6mths         int64
delinq.2yrs            int64
pub.rec                int64
not.fully.paid         int64
dtype: object

── First 5 Rows ──
   credit.policy           purpose  int.rate  installment  log.annual.inc  \
0              1       credit_card    0.2280       580.87         10.8328   
1              1  home_improvement    0.1006       146.06         11.8774   
2              1       credit_card    0.1706       266.78         10.6005   
3              1    small_business    0.0650       201.19         10.6564   
4              0    major_